# Phase 1: Baseline IT Resource Allocation Model

## 1. Problem Definition
In this initial phase, we are solving a static, continuous resource allocation problem. Our goal is to assign employee hours to specific project tasks over a 1-year period to minimize total payroll costs. 

To ensure realism, an employee can only be assigned to a task if their `Primary_Skill` matches the task's `Required_Skill`. 

## 2. Mathematical Formulation
Let us define the components of our Linear Programming (LP) model:

**Sets:**
* $E$: Set of all employees, indexed by $i$.
* $T$: Set of all project tasks, indexed by $j$.
* $V$: Set of all valid assignment pairs $(i,j)$ where employee $i$'s skill matches task $j$'s required skill.

**Parameters:**
* $c_i$: Hourly cost of employee $i$.
* $k_i$: Maximum annual available hours for employee $i$ (e.g., 2080).
* $d_j$: Total hours demanded by task $j$.

**Decision Variables:**
* $x_{ij}$: The continuous number of hours employee $i$ is assigned to task $j$. Defined only for $(i,j) \in V$.

**Objective Function:**
Minimize the total cost of task assignments:
$$\min Z = \sum_{(i,j) \in V} c_i \cdot x_{ij}$$

**Constraints:**
1. **Demand Satisfaction:** Every task must receive exactly the hours it requires.
$$\sum_{i \mid (i,j) \in V} x_{ij} = d_j \quad \forall j \in T$$

2. **Capacity Limit:** No employee can exceed their maximum available hours.
$$\sum_{j \mid (i,j) \in V} x_{ij} \le k_i \quad \forall i \in E$$

3. **Non-Negativity:** Hours assigned cannot be negative.
$$x_{ij} \ge 0 \quad \forall (i,j) \in V$$

In [3]:
# Install pulp if you haven't already: !pip install pulp
import pandas as pd
import numpy as np
import pulp
import random

## 3. Synthetic Data Generation
Instead of relying on static CSVs, we generate our own realistic dataset. This allows us to easily scale the problem size in future phases to test the solver's performance.

We will generate:
1. **10 Employees** with random skills (Backend, Frontend, QA), costs, and standard 2080-hour capacities.
2. **5 Projects** broken down into specific skill-based tasks.

In [9]:
# Set seed for reproducibility
# random.seed(42)
# np.random.seed(42)

skills = ['Backend', 'Frontend', 'QA']

# 1. Generate Employees
employee_data = []
for i in range(1, 11):
    emp_id = f"E{i:02d}"
    skill = random.choice(skills)
    # Base cost depends on skill to add variation
    base_cost = 70 if skill == 'Backend' else (60 if skill == 'Frontend' else 40)
    cost = base_cost + random.randint(-10, 20) 
    
    employee_data.append({
        'Employee_ID': emp_id,
        'Primary_Skill': skill,
        'Hourly_Cost': cost,
        'Max_Hours': 2080
    })

employees_df = pd.DataFrame(employee_data)

# 2. Generate Tasks (Demand)
task_data = []
project_count = 5

for p in range(1, project_count + 1):
    proj_id = f"P{p:02d}"
    # Randomly decide which roles this project needs
    roles_needed = random.sample(skills, k=random.randint(2, 3)) 
    
    for role in roles_needed:
        task_id = f"{proj_id}_{role}"
        # Generate between 200 and 1000 hours needed per task
        hours = random.randint(2, 10) * 100 
        
        task_data.append({
            'Task_ID': task_id,
            'Project_ID': proj_id,
            'Required_Skill': role,
            'Hours_Needed': hours
        })

tasks_df = pd.DataFrame(task_data)

print("--- Employees ---")
display(employees_df.head(5))
print("\n--- Tasks ---")
display(tasks_df.head(5))

--- Employees ---


,Employee_ID,Primary_Skill,Hourly_Cost,Max_Hours
0,E01,QA,49,2080
1,E02,Frontend,68,2080
2,E03,Backend,82,2080
3,E04,Backend,61,2080
4,E05,QA,37,2080



--- Tasks ---


,Task_ID,Project_ID,Required_Skill,Hours_Needed
0,P01_Backend,P01,Backend,500
1,P01_Frontend,P01,Frontend,600
2,P01_QA,P01,QA,300
3,P02_QA,P02,QA,400
4,P02_Backend,P02,Backend,900


## 4. Building and Solving the Model with PuLP
Here, we translate our mathematical equations into Python using the `pulp` library. 

A critical step is filtering for **Valid Pairs**. We only create a decision variable $x_{ij}$ if the employee possesses the skill required for the task. This drastically reduces the computational size of the model and guarantees no unqualified assignments.

In [11]:
from ortools.linear_solver import pywraplp

# Initialize the OR-Tools solver using GLOP (Google's Linear Programming solver)
solver = pywraplp.Solver.CreateSolver('GLOP')

# Create a list of Valid Pairs (i, j) based on skill matching
valid_pairs = []
for _, emp in employees_df.iterrows():
    for _, task in tasks_df.iterrows():
        if emp['Primary_Skill'] == task['Required_Skill']:
            valid_pairs.append((emp['Employee_ID'], task['Task_ID']))

# Define Decision Variables: x_ij (continuous, lower bound = 0)
x = {}
for (i, j) in valid_pairs:
    x[(i, j)] = solver.NumVar(0, solver.infinity(), f'x_{i}_{j}')

# Define Objective Function: Minimize Cost (Hours * Hourly_Cost)
cost_dict = employees_df.set_index('Employee_ID')['Hourly_Cost'].to_dict()
objective = solver.Objective()
for (i, j) in valid_pairs:
    objective.SetCoefficient(x[(i, j)], float(cost_dict[i]))
objective.SetMinimization()

# Constraint 1: Demand Satisfaction (Task must get exact hours needed)
demand_dict = tasks_df.set_index('Task_ID')['Hours_Needed'].to_dict()
for j in tasks_df['Task_ID']:
    capable_employees = [i for (i, t_id) in valid_pairs if t_id == j]
    constraint = solver.Constraint(float(demand_dict[j]), float(demand_dict[j]), f"Demand_{j}")
    for i in capable_employees:
        constraint.SetCoefficient(x[(i, j)], 1)

# Constraint 2: Capacity Limit (Employee cannot exceed Max_Hours)
cap_dict = employees_df.set_index('Employee_ID')['Max_Hours'].to_dict()
for i in employees_df['Employee_ID']:
    assigned_tasks = [j for (e_id, j) in valid_pairs if e_id == i]
    # Lower bound 0, upper bound is their max capacity
    constraint = solver.Constraint(0, float(cap_dict[i]), f"Capacity_{i}")
    for j in assigned_tasks:
        constraint.SetCoefficient(x[(i, j)], 1)

# Solve the model
status = solver.Solve()

if status == pywraplp.Solver.OPTIMAL:
    print("Status: OPTIMAL")
    print(f"Total Optimized Cost: ${solver.Objective().Value():,.2f}")
else:
    print("The problem does not have an optimal solution.")

Status: OPTIMAL
Total Optimized Cost: $358,720.00


In [10]:
# Calculate Total Supply by Skill
supply = employees_df.groupby('Primary_Skill')['Max_Hours'].sum()

# Calculate Total Demand by Skill
demand = tasks_df.groupby('Required_Skill')['Hours_Needed'].sum()

# Combine and Compare
capacity_check = pd.DataFrame({'Total_Supply_Hours': supply, 'Total_Demand_Hours': demand}).fillna(0)
capacity_check['Difference (Supply - Demand)'] = capacity_check['Total_Supply_Hours'] - capacity_check['Total_Demand_Hours']

print("--- Supply vs. Demand Check ---")
display(capacity_check)

--- Supply vs. Demand Check ---


,Total_Supply_Hours,Total_Demand_Hours,Difference (Supply - Demand)
Backend,8320,2900,5420
Frontend,6240,1900,4340
QA,6240,2000,4240


## 5. Results Extraction
The solver has found the optimal allocation. We will now extract the non-zero variables to see exactly who is working on what, and verify that the math constraints held up.

In [12]:
# Extract the results into a readable format
results = []
for (i, j) in valid_pairs:
    assigned_hours = x[(i, j)].solution_value()
    if assigned_hours > 0:  # Only show actual assignments
        cost = assigned_hours * cost_dict[i]
        results.append({
            'Employee_ID': i,
            'Task_ID': j,
            'Assigned_Hours': assigned_hours,
            'Cost_Generated': cost
        })

results_df = pd.DataFrame(results)

print("--- Optimal Assignment Schedule ---")
display(results_df.sort_values(by=['Task_ID', 'Employee_ID']).reset_index(drop=True))

# Sanity Check: Did any employee exceed 2080 hours?
print("\n--- Employee Utilization Check ---")
utilization = results_df.groupby('Employee_ID')['Assigned_Hours'].sum().reset_index()
utilization = utilization.merge(employees_df[['Employee_ID', 'Max_Hours']], on='Employee_ID')
utilization['Utilization_%'] = (utilization['Assigned_Hours'] / utilization['Max_Hours'] * 100).round(1)
display(utilization)

--- Optimal Assignment Schedule ---


,Employee_ID,Task_ID,Assigned_Hours,Cost_Generated
0,E04,P01_Backend,500.0,30500.0
1,E06,P01_Frontend,600.0,31200.0
2,E05,P01_QA,300.0,11100.0
3,E04,P02_Backend,900.0,54900.0
4,E05,P02_QA,400.0,14800.0
5,E04,P03_Backend,500.0,30500.0
6,E06,P03_Frontend,200.0,10400.0
7,E05,P03_QA,200.0,7400.0
8,E04,P04_Backend,180.0,10980.0
9,E08,P04_Backend,320.0,23040.0



--- Employee Utilization Check ---


,Employee_ID,Assigned_Hours,Max_Hours,Utilization_%
0,E04,2080.0,2080,100.0
1,E05,2000.0,2080,96.2
2,E06,1900.0,2080,91.3
3,E08,820.0,2080,39.4
